In [ ]:
from langchain_core.document_loaders.base import BaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage,SystemMessage
import os
import yt_dlp
from pathlib import Path
import webvtt

### Custom Document Loader For fetching Transcript From YouTube Videos


In [ ]:
# Creating a custom document loader that fetches the transcript from the youtube video using yt-dlp and parse it using webvtt

class TranscriptLoader(BaseLoader):

    def __init__(self, video_url:str, language:str="en"):
        self.video_url=video_url
        self.language=language
        self.temp_filename="temp_transcript"
    

    def load(self)->list[Document]:

        # configure the ydl library to get the desired output
        ydl_opts = {
        
        # 'cookiefile':'your-youtube-cookies.txt file',

        # only uncomment this if your model is not running or asking for captcha verification, to do this you must download the cookies.txt file by any cookies.txt locally downloader.

        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': [self.language],
        'subtitlesformat': 'vtt',
        'outtmpl': self.temp_filename,
        'quiet': True,
        'no_warnings': True
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:

            ydl.download([self.video_url])

        expected_file=f"{self.temp_filename}.{self.language}.vtt"
        

        clean_line=[]
        if Path(expected_file).exists():
            for captions in webvtt.read(expected_file):
                lines=captions.text.strip().split('\n')

                # Sometimes youtube combines lines together with a \n so the first step is to ensure that every text senetence is being split into each individual sentence.

                for line in lines:
                    line=line.strip()
                    if not line:
                        continue

                    if not clean_line:
                        clean_line.append(line)

                        #  If the list is empty than directly append the line into the list

                    else:
                        last_line=clean_line[-1]

                        if line==last_line:
                            continue

                        # Checking if the new line and the previous line from the library are same than continue and do nothing. 

                        if line.startswith(last_line):
                            clean_line[-1]=line

                        # check if the new line starts with last line than replace the last line with new line
                        elif last_line not in line:
                            clean_line.append(line)

                        # Else append new line directly.

            os.remove(expected_file)
        else:
            raise FileNotFoundError(f"Transcript is not available for the provided video {self.video_url}")
        
        metadata={"video_url":self.video_url, "language":self.language}

        transcript_text=" ".join(clean_line)

        return [Document(
            page_content=transcript_text.strip(),
            metadata=metadata
        )]
        





In [ ]:
transcript=TranscriptLoader(video_url="https://youtu.be/-gy78kmy0Jg?si=ps8nWEnxnGenl6dl", language="en")
result=transcript.load()

In [ ]:
print(result[0].page_content)

## Splitting The Text Into Sentences

In [ ]:

splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([result[0].page_content])

In [ ]:
chunks

## Initializing the chroma vector store and creating the retriever

In [ ]:
model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',model_kwargs={'device':0})

vectorstore=Chroma.from_documents(
    embedding=model,
    documents=chunks

)

### Retrieving the relevant chunks from the vector store and passing it to the LLM for generating the answer

In [ ]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":10}
)

In [ ]:
def generatecontext(data):
    context="\n\n".join(docs.page_content for docs in data)
    return context


In [ ]:
message=[]

In [ ]:
query="Give me a brief summary "
message.append(HumanMessage(query))

raw_context=retriever.invoke(query)


In [ ]:
context=generatecontext(raw_context)
message.append(SystemMessage(context))

### Designing the prompt template for the LLM to generate the answer based on the retrieved chunks from the vector store

In [ ]:
model=GoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [ ]:
result=model.invoke(message)
result